# Rice Leaf Disease XAI: Revision Experiments (R1)

This notebook regenerates every result needed for the revised manuscript
"Advancing Food Security: XAI-Guided Deep Learning for Interpretable Rice Leaf Disease Classification".

**How to run on Kaggle**
1. Settings: Accelerator **GPU T4 x1** (one GPU is enough), Internet **ON** (pretrained weights are downloaded).
2. Add input: competition **paddy-disease-classification** (you must have joined the competition / accepted its rules).
3. First run with `QUICK_TEST = True` (about 10 minutes) to check that everything works end to end.
4. Then set `QUICK_TEST = False` and use **Save Version -> Save & Run All (Commit)** so the run continues in the background.
5. When it finishes, download `/kaggle/working/results_pack.zip` from the Output tab.

**What it does (in order)**: provenance and MD5 de-duplication -> stratified 75/10/15 split on unique images
(before any augmentation) -> on-the-fly augmentation for the training partition only -> seven-model comparison ->
ResNet50 refinement experiment (baseline / Top Losses removal / random removal, five seeds) -> statistics
(exact McNemar, paired bootstrap, Wilcoxon, two-proportion z-test) -> per-class tables, confusion matrices,
removed-image figure, Grad-CAM figure -> parameter/latency table -> `results_pack.zip`.

The run is resumable: finished runs are stored in `/kaggle/working/results/runs` and skipped if the notebook is restarted.
Results are reported as they come out. The accuracy may be lower than in the first submission and the
Top Losses gain may disappear; the notebook does not tune anything to favour a result.

In [1]:
# ============================== CONFIG (edit here only) ==============================
QUICK_TEST = False          # True = tiny smoke test (few images per class, 1 epoch, 2 seeds)

SEEDS_MAIN = [42, 43, 44, 45, 46]   # seeds for the ResNet50 refinement experiment
SEEDS_COMPARE = [42]                 # seeds for the seven-model comparison
EPOCHS = 12
PATIENCE = 4                # early stopping on validation loss
BATCH_SIZE = 32
IMG_SIZE = 224
LR = 1e-4
N_REMOVE = 5                # images removed by Top Losses and by random removal
SPLIT_SEED = 42             # one fixed split shared by every run (needed for paired tests)
SPLIT_FRACS = (0.75, 0.10, 0.15)
N_BOOT = 5000               # paired bootstrap resamples
LATENCY_WARMUP = 10
LATENCY_RUNS = 30
NUM_WORKERS = 4
USE_PRETRAINED = True       # keep True on Kaggle (ImageNet weights for all models except the DCNN)
NEAR_DUP_HAMMING = 4        # dHash distance used only to REPORT near-duplicates across partitions

MODELS = ["AlexNet", "DCNN", "MobileNetV2", "GoogLeNet", "VGG16", "Xception", "ResNet50"]

DATA_ROOT = None            # None = search /kaggle/input automatically for the train_images folder
WORK_DIR = "/kaggle/working"

if QUICK_TEST:
    SEEDS_MAIN = [42, 43]
    EPOCHS = 1
    PATIENCE = 1
    N_BOOT = 200
    LATENCY_RUNS = 5
    LATENCY_WARMUP = 2
    QUICK_PER_CLASS = 40    # images per class in the smoke test
# =====================================================================================

In [2]:
import os, sys, glob, json, time, math, random, hashlib, shutil, zipfile, platform, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from scipy import stats
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
import timm

warnings.filterwarnings("ignore", category=UserWarning)

# smoke-test outputs never mix with the real run
RESULTS = os.path.join(WORK_DIR, "results_QUICKTEST" if QUICK_TEST else "results")
RUNS_DIR = os.path.join(RESULTS, "runs")
FIG_DIR = os.path.join(RESULTS, "figures")
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints_QUICKTEST" if QUICK_TEST else "checkpoints")   # not zipped
for d in (RESULTS, RUNS_DIR, FIG_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True   # faster; runs are seeded but not bit-exact on GPU

ENV = {
    "python": sys.version.split()[0], "torch": torch.__version__, "torchvision": torchvision.__version__,
    "timm": timm.__version__, "numpy": np.__version__, "platform": platform.platform(),
    "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none",
    "quick_test": QUICK_TEST,
}
print(json.dumps(ENV, indent=2))
T_START = time.time()


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("saved", path)

{
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "timm": "1.0.30",
  "numpy": "2.1.3",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "device": "cuda",
  "gpu": "Tesla T4",
  "quick_test": false
}


## 1. Provenance: locate the data, hash every file, remove exact duplicates

In [3]:
def find_train_root():
    if DATA_ROOT:
        return DATA_ROOT
    hits = sorted(glob.glob("/kaggle/input/**/train_images", recursive=True))
    if not hits:
        raise FileNotFoundError("train_images not found under /kaggle/input. Add the competition "
                                "'paddy-disease-classification' as input (Add Input -> Competitions).")
    return hits[0]

TRAIN_ROOT = find_train_root()
CLASSES = sorted(d for d in os.listdir(TRAIN_ROOT) if os.path.isdir(os.path.join(TRAIN_ROOT, d)))
EXPECTED = ["bacterial_leaf_blight", "bacterial_leaf_streak", "bacterial_panicle_blight", "blast", "brown_spot",
            "dead_heart", "downy_mildew", "hispa", "normal", "tungro"]
print("train root:", TRAIN_ROOT)
print("classes found:", CLASSES)
assert CLASSES == EXPECTED, f"Unexpected class folders: {CLASSES}"
NUM_CLASSES = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}

rows = []
for c in CLASSES:
    for f in sorted(os.listdir(os.path.join(TRAIN_ROOT, c))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append({"image_id": f, "class": c, "path": os.path.join(TRAIN_ROOT, c, f)})
raw = pd.DataFrame(rows)
N_SRC = len(raw)
print("source files:", N_SRC)

# metadata from train.csv (variety, age) if present, for the provenance table
csv_path = os.path.join(os.path.dirname(TRAIN_ROOT), "train.csv")
if os.path.exists(csv_path):
    meta = pd.read_csv(csv_path)
    raw = raw.merge(meta.rename(columns={"label": "csv_label"}), on="image_id", how="left")
    mism = raw[raw["csv_label"].notna() & (raw["csv_label"] != raw["class"])]
    print("folder/csv label mismatches:", len(mism))
    mism.to_csv(os.path.join(RESULTS, "folder_vs_csv_label_mismatches.csv"), index=False)


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

with ThreadPoolExecutor(8) as ex:
    raw["md5"] = list(ex.map(md5_of, raw["path"]))

# image_id is not unique across class folders in every release, so use class/file as the key
raw["uid"] = raw["class"] + "/" + raw["image_id"]
assert raw["uid"].is_unique

grp = raw.groupby("md5")
dup_rows = raw[raw["md5"].duplicated(keep=False)].sort_values(["md5", "uid"]).copy()
dup_rows["n_copies"] = dup_rows.groupby("md5")["uid"].transform("size")
dup_rows["n_labels"] = dup_rows.groupby("md5")["class"].transform("nunique")
dup_rows.to_csv(os.path.join(RESULTS, "duplicates_found.csv"), index=False)

# rule: identical files with one label -> keep the first copy; identical files with conflicting labels -> drop all copies
conflict_md5 = set(dup_rows.loc[dup_rows["n_labels"] > 1, "md5"])
keep = raw[~raw["md5"].isin(conflict_md5)].drop_duplicates("md5", keep="first").reset_index(drop=True)
N_DUP = N_SRC - len(keep)
N_CONFLICT_FILES = int(raw["md5"].isin(conflict_md5).sum())
print(f"exact duplicate groups: {dup_rows['md5'].nunique()}, files removed: {N_DUP} "
      f"(of which in label-conflict groups: {N_CONFLICT_FILES})")

prov = (raw.groupby("class").size().rename("source_files").to_frame()
        .join(keep.groupby("class").size().rename("unique_kept")))
prov["removed_duplicates"] = prov["source_files"] - prov["unique_kept"]
prov.loc["TOTAL"] = prov.sum()
prov.to_csv(os.path.join(RESULTS, "table_provenance_per_class.csv"))
print(prov)

if QUICK_TEST:
    keep = keep.sample(frac=1.0, random_state=0)
    keep = keep[keep.groupby("class").cumcount() < QUICK_PER_CLASS].sort_values("uid").reset_index(drop=True)
    print("QUICK_TEST subset:", len(keep))
data = keep.copy()
data["y"] = data["class"].map(CLS2IDX)
N_UNIQUE = len(data)

train root: /kaggle/input/competitions/paddy-disease-classification/train_images
classes found: ['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']
source files: 10407
folder/csv label mismatches: 0
exact duplicate groups: 72, files removed: 74 (of which in label-conflict groups: 0)
                          source_files  unique_kept  removed_duplicates
class                                                                  
bacterial_leaf_blight              479          471                   8
bacterial_leaf_streak              380          380                   0
bacterial_panicle_blight           337          336                   1
blast                             1738         1728                  10
brown_spot                         965          953                  12
dead_heart                        1442         1429                  13
downy_mildew                   

## 2. Stratified split on unique images, BEFORE any augmentation

In [4]:
tr_f, va_f, te_f = SPLIT_FRACS
idx_all = np.arange(N_UNIQUE)
idx_trva, idx_te = train_test_split(idx_all, test_size=te_f, stratify=data["y"], random_state=SPLIT_SEED)
idx_tr, idx_va = train_test_split(idx_trva, test_size=va_f / (tr_f + va_f),
                                  stratify=data["y"].values[idx_trva], random_state=SPLIT_SEED)
data["partition"] = ""
data.loc[idx_tr, "partition"] = "train"
data.loc[idx_va, "partition"] = "val"
data.loc[idx_te, "partition"] = "test"

S_tr, S_va, S_te = set(data.uid[idx_tr]), set(data.uid[idx_va]), set(data.uid[idx_te])
assert not (S_tr & S_va) and not (S_tr & S_te) and not (S_va & S_te), "partitions overlap"
M_tr, M_va, M_te = set(data.md5[idx_tr]), set(data.md5[idx_va]), set(data.md5[idx_te])
assert not (M_tr & M_va) and not (M_tr & M_te) and not (M_va & M_te), "identical file in two partitions"
assert len(S_tr) + len(S_va) + len(S_te) == N_UNIQUE
print("partitions are disjoint (by id and by MD5)")

data[["uid", "image_id", "class", "y", "partition", "md5"]].to_csv(os.path.join(RESULTS, "split_manifest.csv"), index=False)

tab = pd.crosstab(data["class"], data["partition"])[["train", "val", "test"]]
tab["total"] = tab.sum(axis=1)
tab.loc["TOTAL"] = tab.sum()
tab.to_csv(os.path.join(RESULTS, "table_unique_images_per_class_and_partition.csv"))
print(tab)
N_TRAIN, N_VAL, N_TEST = len(idx_tr), len(idx_va), len(idx_te)

# Figure 2: count-labelled bar chart (unique images per class, stacked by partition)
fig, ax = plt.subplots(figsize=(10, 4.8))
t = tab.drop(index="TOTAL")
order = t["total"].sort_values(ascending=False).index
bottom = np.zeros(len(order))
for part, col in zip(["train", "val", "test"], ["#4C72B0", "#DD8452", "#55A868"]):
    vals = t.loc[order, part].values
    ax.bar(range(len(order)), vals, bottom=bottom, color=col, label=part)
    bottom += vals
for i, v in enumerate(t.loc[order, "total"].values):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([c.replace("_", " ") for c in order], rotation=35, ha="right")
ax.set_ylabel("Unique images")
ax.set_title(f"Unique images per class and partition (n = {N_UNIQUE})")
ax.legend(frameon=False, loc="upper right")
save_fig(fig, "fig2_class_counts.png")

partitions are disjoint (by id and by MD5)
partition                 train   val  test  total
class                                             
bacterial_leaf_blight       353    47    71    471
bacterial_leaf_streak       285    38    57    380
bacterial_panicle_blight    251    34    51    336
blast                      1296   173   259   1728
brown_spot                  715    95   143    953
dead_heart                 1072   143   214   1429
downy_mildew                463    62    93    618
hispa                      1192   159   238   1589
normal                     1312   175   262   1749
tungro                      810   108   162   1080
TOTAL                      7749  1034  1550  10333
saved /kaggle/working/results/figures/fig2_class_counts.png


### 2b. Near-duplicate report (perceptual hash; report only, nothing is removed)

In [5]:
def dhash(path, size=8):
    im = Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return np.packbits(bits).view(">u8")[0]

with ThreadPoolExecutor(8) as ex:
    hashes = np.array(list(ex.map(dhash, data["path"])), dtype=np.uint64)

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
part_arr = data["partition"].values
pairs = []
CH = 512
for s in range(0, N_UNIQUE, CH):
    x = hashes[s:s + CH, None] ^ hashes[None, :]
    d = POP[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1)
    ii, jj = np.nonzero(d <= NEAR_DUP_HAMMING)
    ii = ii + s
    m = (jj > ii) & (part_arr[ii] != part_arr[jj])
    for a, b in zip(ii[m], jj[m]):
        pairs.append((data.uid[a], data.partition[a], data.uid[b], data.partition[b], int(d[a - s, b])))
near = pd.DataFrame(pairs, columns=["uid_a", "partition_a", "uid_b", "partition_b", "hamming"])
near.to_csv(os.path.join(RESULTS, "near_duplicates_cross_partition.csv"), index=False)
N_NEAR_CROSS = len(near)
print(f"near-duplicate pairs across partitions (dHash distance <= {NEAR_DUP_HAMMING}): {N_NEAR_CROSS}")

near-duplicate pairs across partitions (dHash distance <= 4): 1019


## 3. Image cache and data loaders (augmentation on the training partition only)

In [6]:
def load_resized(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    CACHE = np.stack(list(ex.map(load_resized, data["path"])))   # (N, H, W, 3) uint8, about 1.6 GB for 10k images
print("image cache:", CACHE.shape, f"{CACHE.nbytes / 1e9:.2f} GB in {time.time() - t0:.0f}s")
Y = data["y"].values.astype(np.int64)
UIDS = data["uid"].values

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

TRAIN_TF = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), antialias=True),
    T.ColorJitter(brightness=0.2),
])


class CachedDS(Dataset):
    def __init__(self, indices, train):
        self.idx = np.asarray(indices)
        self.train = train

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        x = torch.from_numpy(CACHE[i]).permute(2, 0, 1).contiguous()   # uint8 CHW
        if self.train:
            x = TRAIN_TF(x)
        x = (x.float() / 255.0 - MEAN) / STD
        return x, Y[i], i


def make_loader(indices, train, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(CachedDS(indices, train), batch_size=BATCH_SIZE, shuffle=train, generator=g,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      multiprocessing_context="fork" if NUM_WORKERS > 0 else None,
                      persistent_workers=False, drop_last=False)

IDX_TR = np.sort(idx_tr); IDX_VA = np.sort(idx_va); IDX_TE = np.sort(idx_te)

image cache: (10333, 224, 224, 3) 1.56 GB in 23s


## 4. Models (all with a ten-class head)

In [7]:
class DCNN(nn.Module):
    """Custom DCNN trained from scratch: 4 blocks of [3x3 conv-BN-ReLU] x2 + max pool (32/64/128/256),
    global average pooling, dropout 0.3, linear classifier."""
    def __init__(self, n_classes, widths=(32, 64, 128, 256)):
        super().__init__()
        layers, c_in = [], 3
        for w in widths:
            layers += [nn.Conv2d(c_in, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                       nn.Conv2d(w, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = w
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(c_in, n_classes)

    def forward(self, x):
        return self.fc(self.drop(torch.flatten(self.pool(self.features(x)), 1)))


def build_model(name, n=NUM_CLASSES):
    tvm = torchvision.models
    W = USE_PRETRAINED
    if name == "AlexNet":
        m = tvm.alexnet(weights=tvm.AlexNet_Weights.IMAGENET1K_V1 if W else None)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, n)
    elif name == "DCNN":
        m = DCNN(n)
    elif name == "MobileNetV2":
        m = tvm.mobilenet_v2(weights=tvm.MobileNet_V2_Weights.IMAGENET1K_V1 if W else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "GoogLeNet":
        if W:
            m = tvm.googlenet(weights=tvm.GoogLeNet_Weights.IMAGENET1K_V1)
        else:
            m = tvm.googlenet(weights=None, aux_logits=False, init_weights=True)
        m.aux_logits = False
        m.aux1 = None
        m.aux2 = None
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "VGG16":
        m = tvm.vgg16(weights=tvm.VGG16_Weights.IMAGENET1K_V1 if W else None)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, n)
    elif name == "Xception":
        try:
            m = timm.create_model("legacy_xception", pretrained=W, num_classes=n)
        except Exception as e:
            print("legacy_xception unavailable, falling back to 'xception':", e)
            m = timm.create_model("xception", pretrained=W, num_classes=n)
    elif name == "ResNet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2 if W else None)
        m.fc = nn.Linear(m.fc.in_features, n)
    else:
        raise ValueError(name)
    return m


def n_params_m(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

## 5. Training, evaluation and metrics

In [8]:
def confusion(y_true, y_pred, n=NUM_CLASSES):
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm


def metrics_from_cm(cm):
    """Per-class one-vs-rest metrics and macro averages.
    TN for class c = images that are neither of class c nor predicted as c."""
    cm = np.asarray(cm, dtype=np.int64)
    n = cm.sum()
    tp = np.diag(cm).astype(float)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    tn = n - tp - fp - fn
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(tp + fp > 0, tp / (tp + fp), 0.0)
        rec = np.where(tp + fn > 0, tp / (tp + fn), 0.0)
        spec = np.where(tn + fp > 0, tn / (tn + fp), 0.0)
        f1 = np.where(prec + rec > 0, 2 * prec * rec / (prec + rec), 0.0)
    per_class = pd.DataFrame({"class": CLASSES, "support": cm.sum(1), "TP": tp.astype(int), "FP": fp.astype(int),
                              "FN": fn.astype(int), "TN": tn.astype(int), "precision": prec, "recall": rec,
                              "specificity": spec, "f1": f1})
    correct = int(tp.sum())
    summary = {"n": int(n), "correct": correct, "accuracy": correct / n,
               "macro_precision": prec.mean(), "macro_recall": rec.mean(),
               "macro_specificity": spec.mean(), "macro_f1": f1.mean()}
    return summary, per_class


@torch.no_grad()
def predict(model, indices):
    model.eval()
    out_p, out_i = [], []
    for x, _, i in make_loader(indices, train=False):
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        out_p.append(F.softmax(logits.float(), 1).cpu().numpy())
        out_i.append(i.numpy())
    return np.concatenate(out_p), np.concatenate(out_i)


def train_one(name, seed, train_idx):
    set_seed(seed)
    model = build_model(name).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    crit = nn.CrossEntropyLoss()
    tr_loader = make_loader(train_idx, train=True, seed=seed)
    va_loader = make_loader(IDX_VA, train=False)
    best, best_state, bad, hist = float("inf"), None, 0, []
    for ep in range(1, EPOCHS + 1):
        t0 = time.time(); model.train(); tl, tn = 0.0, 0
        for x, y, _ in tr_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tn += len(y)
        sched.step()
        model.eval(); vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for x, y, _ in va_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                    lo = model(x)
                vl += crit(lo.float(), y).item() * len(y); vc += (lo.argmax(1) == y).sum().item(); vn += len(y)
        vl /= vn
        hist.append({"epoch": ep, "train_loss": tl / tn, "val_loss": vl, "val_acc": vc / vn, "sec": time.time() - t0})
        print(f"  [{name} s{seed}] ep {ep:02d} train {tl/tn:.4f} val {vl:.4f} acc {vc/vn:.4f} ({time.time()-t0:.0f}s)")
        if vl < best - 1e-6:
            best, bad = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  early stop at epoch {ep}")
                break
    model.load_state_dict(best_state)
    best_ep = int(np.argmin([h["val_loss"] for h in hist]) + 1)
    return model, hist, best_ep


def config_fingerprint(train_idx):
    """Everything that changes a run's result; a stored run is reused only if this matches."""
    key = json.dumps({"q": QUICK_TEST, "ep": EPOCHS, "pat": PATIENCE, "bs": BATCH_SIZE, "img": IMG_SIZE, "lr": LR,
                      "pre": USE_PRETRAINED, "split": SPLIT_SEED, "train": hashlib.md5(
                          ",".join(sorted(UIDS[np.asarray(train_idx)])).encode()).hexdigest(),
                      "test": hashlib.md5(",".join(sorted(UIDS[IDX_TE])).encode()).hexdigest()})
    return hashlib.md5(key.encode()).hexdigest()


def run_or_load(tag, name, seed, train_idx, save_ckpt=False):
    """Train (or reload a finished run), evaluate on the test set, and store everything under runs/<tag>."""
    mfile = os.path.join(RUNS_DIR, f"{tag}_metrics.json")
    ckpt = os.path.join(CKPT_DIR, f"{tag}.pt")
    fp = config_fingerprint(train_idx)
    if os.path.exists(mfile) and (not save_ckpt or os.path.exists(ckpt)) and json.load(open(mfile)).get("fingerprint") == fp:
        print(f"[skip] {tag} already finished")
        res = json.load(open(mfile))
        res["cm"] = np.load(os.path.join(RUNS_DIR, f"{tag}_cm.npy"))
        res["pred_df"] = pd.read_csv(os.path.join(RUNS_DIR, f"{tag}_predictions.csv"))
        return res
    print(f"[train] {tag}")
    t0 = time.time()
    model, hist, best_ep = train_one(name, seed, train_idx)
    probs, idx = predict(model, IDX_TE)
    y_true, y_pred = Y[idx], probs.argmax(1)
    cm = confusion(y_true, y_pred)
    summ, per_class = metrics_from_cm(cm)
    pred_df = pd.DataFrame({"uid": UIDS[idx], "y_true": y_true, "y_pred": y_pred,
                            "true_label": [CLASSES[k] for k in y_true], "pred_label": [CLASSES[k] for k in y_pred],
                            "confidence": probs.max(1), "correct": (y_true == y_pred).astype(int)})
    pred_df.to_csv(os.path.join(RUNS_DIR, f"{tag}_predictions.csv"), index=False)
    per_class.to_csv(os.path.join(RUNS_DIR, f"{tag}_per_class.csv"), index=False)
    pd.DataFrame(hist).to_csv(os.path.join(RUNS_DIR, f"{tag}_history.csv"), index=False)
    np.save(os.path.join(RUNS_DIR, f"{tag}_cm.npy"), cm)
    res = {"tag": tag, "model": name, "seed": seed, "n_train": int(len(train_idx)), "best_epoch": best_ep,
           "epochs_run": len(hist), "train_minutes": (time.time() - t0) / 60, "fingerprint": fp, **summ}
    json.dump(res, open(mfile, "w"), indent=2, default=float)
    if save_ckpt:
        torch.save(model.state_dict(), ckpt)
    print(f"  -> {tag}: {summ['correct']}/{summ['n']} = {summ['accuracy']:.4f}, macro F1 {summ['macro_f1']:.4f}")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    res["cm"], res["pred_df"] = cm, pred_df
    return res


def load_model(tag, name="ResNet50"):
    m = build_model(name).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"{tag}.pt"), map_location=DEVICE))
    return m.eval()

## 6. ResNet50 refinement experiment: baseline / Top Losses removal / random removal

In [9]:
@torch.no_grad()
def training_losses(model, indices):
    """Cross-entropy of every training image, eval mode, no augmentation."""
    model.eval()
    rows = []
    for x, y, i in make_loader(indices, train=False):
        x = x.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            lo = model(x).float()
        l = F.cross_entropy(lo, y.to(DEVICE), reduction="none").cpu().numpy()
        p = F.softmax(lo, 1).cpu().numpy()
        for k in range(len(i)):
            rows.append((int(i[k]), float(l[k]), int(p[k].argmax()), float(p[k].max()), float(p[k][y[k]])))
    return pd.DataFrame(rows, columns=["idx", "loss", "y_pred", "pred_conf", "true_conf"])


REF = {}            # (arm, seed) -> result
removed_rows = []
for seed in SEEDS_MAIN:
    base = run_or_load(f"ResNet50_baseline_s{seed}", "ResNet50", seed, IDX_TR, save_ckpt=True)
    REF[("baseline", seed)] = base

    # Top Losses: score with this seed's baseline model, remove the N_REMOVE highest-loss training images
    tl_file = os.path.join(RUNS_DIR, f"toplosses_s{seed}.csv")
    if os.path.exists(tl_file):
        tl = pd.read_csv(tl_file)
    else:
        bm = load_model(f"ResNet50_baseline_s{seed}")
        tl = training_losses(bm, IDX_TR).sort_values("loss", ascending=False).reset_index(drop=True)
        tl["uid"] = UIDS[tl["idx"]]
        tl["true_label"] = [CLASSES[Y[i]] for i in tl["idx"]]
        tl["pred_label"] = [CLASSES[k] for k in tl["y_pred"]]
        tl.head(50).to_csv(tl_file, index=False)          # top 50 kept for inspection
        del bm
    top_idx = tl["idx"].values[:N_REMOVE]
    for r, row in tl.head(N_REMOVE).iterrows():
        removed_rows.append({"seed": seed, "rank": r + 1, "uid": row["uid"], "true_label": row["true_label"],
                             "pred_label": row["pred_label"], "loss": row["loss"], "pred_conf": row["pred_conf"]})
    keep_tl = np.setdiff1d(IDX_TR, top_idx)
    assert len(keep_tl) == len(IDX_TR) - N_REMOVE
    REF[("toplosses", seed)] = run_or_load(f"ResNet50_toplosses_s{seed}", "ResNet50", seed, keep_tl,
                                           save_ckpt=(seed == SEEDS_MAIN[0]))

    # Random removal control: new draw per seed
    rng = np.random.default_rng(seed + 1000)
    rnd_idx = rng.choice(IDX_TR, size=N_REMOVE, replace=False)
    pd.DataFrame({"seed": seed, "uid": UIDS[rnd_idx], "true_label": [CLASSES[Y[i]] for i in rnd_idx]}).to_csv(
        os.path.join(RUNS_DIR, f"random_removed_s{seed}.csv"), index=False)
    keep_rd = np.setdiff1d(IDX_TR, rnd_idx)
    REF[("random", seed)] = run_or_load(f"ResNet50_random_s{seed}", "ResNet50", seed, keep_rd)

removed_df = pd.DataFrame(removed_rows)
removed_df.to_csv(os.path.join(RESULTS, "top_losses_removed_images.csv"), index=False)
print(removed_df)

ARMS = ["baseline", "toplosses", "random"]
ref_rows = []
for (arm, seed), r in REF.items():
    ref_rows.append({"arm": arm, "seed": seed, "n_train": r["n_train"], "correct": r["correct"], "n": r["n"],
                     "accuracy": r["accuracy"], "macro_precision": r["macro_precision"],
                     "macro_recall": r["macro_recall"], "macro_specificity": r["macro_specificity"],
                     "macro_f1": r["macro_f1"], "best_epoch": r["best_epoch"]})
ref_df = pd.DataFrame(ref_rows).sort_values(["seed", "arm"])
ref_df.to_csv(os.path.join(RESULTS, "refinement_runs.csv"), index=False)

agg = ref_df.groupby("arm").agg(
    seeds=("seed", "count"), correct_mean=("correct", "mean"), correct_min=("correct", "min"),
    correct_max=("correct", "max"), acc_mean=("accuracy", "mean"), acc_sd=("accuracy", "std"),
    macro_precision_mean=("macro_precision", "mean"), macro_recall_mean=("macro_recall", "mean"),
    macro_specificity_mean=("macro_specificity", "mean"), macro_f1_mean=("macro_f1", "mean")).reindex(ARMS)
agg.to_csv(os.path.join(RESULTS, "table9_refinement_arms_across_seeds.csv"))
print(agg)

# Table 4: baseline vs Top Losses refined (mean over seeds, plus first seed)
t4 = []
for arm in ["baseline", "toplosses"]:
    sub = ref_df[ref_df.arm == arm]
    first = sub[sub.seed == SEEDS_MAIN[0]].iloc[0]
    t4.append({"model": "ResNet50 " + ("baseline" if arm == "baseline" else "Top Losses refined"),
               f"correct_seed{SEEDS_MAIN[0]}": int(first.correct), "n": int(first.n),
               f"accuracy_seed{SEEDS_MAIN[0]}": first.accuracy,
               "accuracy_mean": sub.accuracy.mean(), "accuracy_sd": sub.accuracy.std(),
               "macro_precision_mean": sub.macro_precision.mean(), "macro_recall_mean": sub.macro_recall.mean(),
               "macro_specificity_mean": sub.macro_specificity.mean(), "macro_f1_mean": sub.macro_f1.mean()})
pd.DataFrame(t4).to_csv(os.path.join(RESULTS, "table4_baseline_vs_refined.csv"), index=False)

[train] ResNet50_baseline_s42
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


  [ResNet50 s42] ep 01 train 1.3444 val 0.6240 acc 0.7940 (39s)
  [ResNet50 s42] ep 02 train 0.4773 val 0.3357 acc 0.9014 (33s)
  [ResNet50 s42] ep 03 train 0.2754 val 0.2190 acc 0.9304 (34s)
  [ResNet50 s42] ep 04 train 0.1737 val 0.1422 acc 0.9584 (35s)
  [ResNet50 s42] ep 05 train 0.1144 val 0.1362 acc 0.9613 (35s)
  [ResNet50 s42] ep 06 train 0.0841 val 0.1481 acc 0.9613 (35s)
  [ResNet50 s42] ep 07 train 0.0706 val 0.1192 acc 0.9671 (35s)
  [ResNet50 s42] ep 08 train 0.0527 val 0.0948 acc 0.9720 (35s)
  [ResNet50 s42] ep 09 train 0.0413 val 0.0946 acc 0.9710 (35s)
  [ResNet50 s42] ep 10 train 0.0398 val 0.0922 acc 0.9710 (35s)
  [ResNet50 s42] ep 11 train 0.0326 val 0.0819 acc 0.9739 (35s)
  [ResNet50 s42] ep 12 train 0.0275 val 0.0858 acc 0.9710 (35s)
  -> ResNet50_baseline_s42: 1512/1550 = 0.9755, macro F1 0.9733
[train] ResNet50_toplosses_s42
  [ResNet50 s42] ep 01 train 1.3321 val 0.6616 acc 0.7737 (35s)
  [ResNet50 s42] ep 02 train 0.5082 val 0.3117 acc 0.9014 (35s)
  [ResNet

## 7. Seven-model comparison (Table 3) and measured parameters / latency (Table 5)

In [10]:
CMP = {}
for name in MODELS:
    for seed in SEEDS_COMPARE:
        if name == "ResNet50" and ("baseline", seed) in REF:
            CMP[(name, seed)] = REF[("baseline", seed)]   # identical configuration, reuse the baseline run
        else:
            CMP[(name, seed)] = run_or_load(f"{name}_s{seed}", name, seed, IDX_TR)

t3 = pd.DataFrame([{"model": n, "seed": s, "correct": r["correct"], "n": r["n"], "accuracy": r["accuracy"],
                    "macro_precision": r["macro_precision"], "macro_recall": r["macro_recall"],
                    "macro_specificity": r["macro_specificity"], "macro_f1": r["macro_f1"],
                    "best_epoch": r["best_epoch"], "epochs_run": r["epochs_run"]}
                   for (n, s), r in CMP.items()])
t3.to_csv(os.path.join(RESULTS, "table3_model_comparison_all_runs.csv"), index=False)
t3s = t3.groupby("model", sort=False).agg(
    seeds=("seed", "count"), correct=("correct", "mean"), n=("n", "first"), accuracy=("accuracy", "mean"),
    macro_precision=("macro_precision", "mean"), macro_recall=("macro_recall", "mean"),
    macro_specificity=("macro_specificity", "mean"), macro_f1=("macro_f1", "mean"))
t3s["correct_over_n"] = [f"{int(round(c))}/{n}" for c, n in zip(t3s.correct, t3s.n)]
t3s = t3s.sort_values("accuracy", ascending=False)
t3s["rank"] = np.arange(1, len(t3s) + 1)
t3s.to_csv(os.path.join(RESULTS, "table3_model_comparison_summary.csv"))
print(t3s)

for (n, s), r in CMP.items():
    pd.read_csv(os.path.join(RUNS_DIR, f"{r['tag']}_per_class.csv")).assign(model=n, seed=s).to_csv(
        os.path.join(RUNS_DIR, f"per_class_{n}_s{s}.csv"), index=False)


@torch.no_grad()
def latency_ms(model):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    for _ in range(LATENCY_WARMUP):
        model(x)
    times = []
    for _ in range(LATENCY_RUNS):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter(); model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times)), float(np.mean(times)), float(np.std(times))

lat_rows = []
for name in MODELS:
    m = build_model(name).to(DEVICE)
    med, mean, sd = latency_ms(m)
    lat_rows.append({"model": name, "params_millions": n_params_m(m), "latency_ms_median": med,
                     "latency_ms_mean": mean, "latency_ms_sd": sd, "batch_size": 1, "precision": "fp32",
                     "input": f"{IMG_SIZE}x{IMG_SIZE}", "timed_runs": LATENCY_RUNS, "device": ENV["gpu"]})
    del m
pd.DataFrame(lat_rows).to_csv(os.path.join(RESULTS, "table5_measured_params_latency.csv"), index=False)
print(pd.DataFrame(lat_rows))

[train] AlexNet_s42
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:00<00:00, 248MB/s]


  [AlexNet s42] ep 01 train 1.3662 val 0.8090 acc 0.7485 (23s)
  [AlexNet s42] ep 02 train 0.7511 val 0.5870 acc 0.8085 (22s)
  [AlexNet s42] ep 03 train 0.5409 val 0.4287 acc 0.8636 (22s)
  [AlexNet s42] ep 04 train 0.4008 val 0.3315 acc 0.8878 (22s)
  [AlexNet s42] ep 05 train 0.2952 val 0.2584 acc 0.9207 (22s)
  [AlexNet s42] ep 06 train 0.2219 val 0.2233 acc 0.9362 (22s)
  [AlexNet s42] ep 07 train 0.1681 val 0.2134 acc 0.9342 (22s)
  [AlexNet s42] ep 08 train 0.1384 val 0.1863 acc 0.9497 (22s)
  [AlexNet s42] ep 09 train 0.1039 val 0.1419 acc 0.9555 (22s)
  [AlexNet s42] ep 10 train 0.0823 val 0.1287 acc 0.9623 (22s)
  [AlexNet s42] ep 11 train 0.0678 val 0.1237 acc 0.9691 (23s)
  [AlexNet s42] ep 12 train 0.0689 val 0.1256 acc 0.9652 (23s)
  -> AlexNet_s42: 1483/1550 = 0.9568, macro F1 0.9528
[train] DCNN_s42
  [DCNN s42] ep 01 train 1.8800 val 1.6496 acc 0.4052 (30s)
  [DCNN s42] ep 02 train 1.6051 val 1.4106 acc 0.5300 (25s)
  [DCNN s42] ep 03 train 1.4357 val 1.5974 acc 0.4642

100%|██████████| 13.6M/13.6M [00:00<00:00, 216MB/s]


  [MobileNetV2 s42] ep 01 train 1.1476 val 0.5715 acc 0.8230 (45s)
  [MobileNetV2 s42] ep 02 train 0.5229 val 0.3500 acc 0.8849 (25s)
  [MobileNetV2 s42] ep 03 train 0.3249 val 0.2860 acc 0.9159 (25s)
  [MobileNetV2 s42] ep 04 train 0.2249 val 0.1911 acc 0.9458 (25s)
  [MobileNetV2 s42] ep 05 train 0.1725 val 0.1554 acc 0.9507 (25s)
  [MobileNetV2 s42] ep 06 train 0.1279 val 0.1563 acc 0.9458 (25s)
  [MobileNetV2 s42] ep 07 train 0.1013 val 0.1302 acc 0.9613 (25s)
  [MobileNetV2 s42] ep 08 train 0.0829 val 0.1070 acc 0.9652 (25s)
  [MobileNetV2 s42] ep 09 train 0.0673 val 0.1064 acc 0.9691 (25s)
  [MobileNetV2 s42] ep 10 train 0.0613 val 0.0900 acc 0.9720 (25s)
  [MobileNetV2 s42] ep 11 train 0.0553 val 0.0906 acc 0.9681 (25s)
  [MobileNetV2 s42] ep 12 train 0.0474 val 0.0864 acc 0.9671 (25s)
  -> MobileNetV2_s42: 1490/1550 = 0.9613, macro F1 0.9561
[train] GoogLeNet_s42
Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/goog

100%|██████████| 49.7M/49.7M [00:00<00:00, 246MB/s]


  [GoogLeNet s42] ep 01 train 1.4174 val 0.8045 acc 0.7505 (28s)
  [GoogLeNet s42] ep 02 train 0.6742 val 0.4834 acc 0.8501 (25s)
  [GoogLeNet s42] ep 03 train 0.4359 val 0.3439 acc 0.8936 (26s)
  [GoogLeNet s42] ep 04 train 0.3017 val 0.2401 acc 0.9304 (26s)
  [GoogLeNet s42] ep 05 train 0.2186 val 0.1675 acc 0.9507 (26s)
  [GoogLeNet s42] ep 06 train 0.1659 val 0.1371 acc 0.9613 (26s)
  [GoogLeNet s42] ep 07 train 0.1249 val 0.1312 acc 0.9642 (26s)
  [GoogLeNet s42] ep 08 train 0.1128 val 0.1198 acc 0.9691 (26s)
  [GoogLeNet s42] ep 09 train 0.0916 val 0.1152 acc 0.9729 (26s)
  [GoogLeNet s42] ep 10 train 0.0824 val 0.1027 acc 0.9729 (26s)
  [GoogLeNet s42] ep 11 train 0.0756 val 0.0965 acc 0.9710 (26s)
  [GoogLeNet s42] ep 12 train 0.0727 val 0.0970 acc 0.9700 (26s)
  -> GoogLeNet_s42: 1490/1550 = 0.9613, macro F1 0.9598
[train] VGG16_s42
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 235MB/s]


  [VGG16 s42] ep 01 train 1.4766 val 0.8704 acc 0.6915 (76s)
  [VGG16 s42] ep 02 train 0.8493 val 0.5964 acc 0.7814 (71s)
  [VGG16 s42] ep 03 train 0.5864 val 0.5113 acc 0.8414 (71s)
  [VGG16 s42] ep 04 train 0.3800 val 0.3665 acc 0.8965 (70s)
  [VGG16 s42] ep 05 train 0.2713 val 0.2913 acc 0.9168 (71s)
  [VGG16 s42] ep 06 train 0.2223 val 0.1959 acc 0.9381 (70s)
  [VGG16 s42] ep 07 train 0.1363 val 0.1843 acc 0.9429 (70s)
  [VGG16 s42] ep 08 train 0.0998 val 0.1688 acc 0.9555 (70s)
  [VGG16 s42] ep 09 train 0.0713 val 0.1399 acc 0.9603 (70s)
  [VGG16 s42] ep 10 train 0.0456 val 0.0973 acc 0.9797 (70s)
  [VGG16 s42] ep 11 train 0.0369 val 0.0946 acc 0.9768 (70s)
  [VGG16 s42] ep 12 train 0.0332 val 0.0926 acc 0.9787 (70s)
  -> VGG16_s42: 1504/1550 = 0.9703, macro F1 0.9659
[train] Xception_s42
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
  [Xception s42]

## 8. Statistics: exact McNemar, paired bootstrap, Wilcoxon, two-proportion z-test

In [11]:
def aligned_correct(r1, r2):
    a = r1["pred_df"].set_index("uid")["correct"]
    b = r2["pred_df"].set_index("uid")["correct"].reindex(a.index)
    assert b.notna().all(), "test sets differ"
    return a.values.astype(int), b.values.astype(int)


def mcnemar_exact(c1, c2):
    b = int(((c1 == 1) & (c2 == 0)).sum())   # first right, second wrong
    c = int(((c1 == 0) & (c2 == 1)).sum())   # first wrong, second right
    p = 1.0 if b + c == 0 else float(stats.binomtest(min(b, c), b + c, 0.5).pvalue)
    return b, c, p


def paired_bootstrap(c1, c2, n_boot=N_BOOT, seed=0):
    rng = np.random.default_rng(seed)
    n = len(c1)
    d = c2.astype(float) - c1.astype(float)
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))
    return float(d.mean()), float(lo), float(hi), float(p)


def two_prop_z(k1, n1, k2, n2):
    p1, p2 = k1 / n1, k2 / n2
    p = (k1 + k2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    z = 0.0 if se == 0 else (p1 - p2) / se
    return z, float(2 * stats.norm.sf(abs(z)))

# Table 6a: per-seed paired tests of each refinement arm against its own baseline
t6 = []
for seed in SEEDS_MAIN:
    for arm in ["toplosses", "random"]:
        c1, c2 = aligned_correct(REF[("baseline", seed)], REF[(arm, seed)])
        b, c, p = mcnemar_exact(c1, c2)
        md, lo, hi, pb = paired_bootstrap(c1, c2, seed=seed)
        t6.append({"seed": seed, "comparison": f"baseline vs {arm}", "baseline_correct": int(c1.sum()),
                   "arm_correct": int(c2.sum()), "n": len(c1), "b_baseline_only": b, "c_arm_only": c,
                   "mcnemar_exact_p": p, "acc_diff": md, "boot_ci_low": lo, "boot_ci_high": hi, "boot_p": pb,
                   "significant_0.05": bool(p < 0.05 and (lo > 0 or hi < 0))})
t6 = pd.DataFrame(t6)
t6.to_csv(os.path.join(RESULTS, "table6_paired_tests_refinement.csv"), index=False)
print(t6)

# Table 6b: across seeds, Top Losses vs random removal (Wilcoxon signed-rank on accuracy gains)
g_tl = np.array([REF[("toplosses", s)]["accuracy"] - REF[("baseline", s)]["accuracy"] for s in SEEDS_MAIN])
g_rd = np.array([REF[("random", s)]["accuracy"] - REF[("baseline", s)]["accuracy"] for s in SEEDS_MAIN])
diff = g_tl - g_rd
if len(SEEDS_MAIN) >= 2 and np.any(diff != 0):
    w = stats.wilcoxon(g_tl, g_rd, zero_method="wilcox", alternative="two-sided")
    w_stat, w_p = float(w.statistic), float(w.pvalue)
else:
    w_stat, w_p = float("nan"), 1.0
min_p = 2 / 2 ** len(SEEDS_MAIN)   # smallest two-sided exact p reachable with this many seeds
t6b = pd.DataFrame([{"test": "Wilcoxon signed-rank, gain(Top Losses) vs gain(random)", "seeds": len(SEEDS_MAIN),
                     "mean_gain_toplosses": g_tl.mean(), "mean_gain_random": g_rd.mean(),
                     "seeds_toplosses_better": int((diff > 0).sum()), "seeds_random_better": int((diff < 0).sum()),
                     "seeds_tied": int((diff == 0).sum()), "statistic": w_stat, "p_value": w_p,
                     "smallest_reachable_p": min_p}])
t6b.to_csv(os.path.join(RESULTS, "table6b_wilcoxon_toplosses_vs_random.csv"), index=False)
print(t6b.T)
print(f"NOTE: with {len(SEEDS_MAIN)} seeds the smallest two-sided Wilcoxon p is {min_p:.4f}; "
      "report the sign pattern alongside the p-value.")

# Table 6c: ResNet50 vs selected models (same test images)
s0 = SEEDS_COMPARE[0]
t6c = []
for other in ["Xception", "MobileNetV2", "DCNN"]:
    r1, r2 = CMP[("ResNet50", s0)], CMP[(other, s0)]
    z, pz = two_prop_z(r1["correct"], r1["n"], r2["correct"], r2["n"])
    c1, c2 = aligned_correct(r2, r1)          # order: other, ResNet50 -> positive diff favours ResNet50
    b, c, pm = mcnemar_exact(c1, c2)
    md, lo, hi, pb = paired_bootstrap(c1, c2, seed=s0)
    t6c.append({"comparison": f"ResNet50 vs {other}", "resnet50_correct": r1["correct"], "other_correct": r2["correct"],
                "n": r1["n"], "acc_diff": r1["accuracy"] - r2["accuracy"], "z": z, "z_test_p": pz,
                "mcnemar_other_only": b, "mcnemar_resnet_only": c, "mcnemar_exact_p": pm,
                "boot_ci_low": lo, "boot_ci_high": hi, "boot_p": pb})
t6c = pd.DataFrame(t6c)
t6c.to_csv(os.path.join(RESULTS, "table6c_model_pair_tests.csv"), index=False)
print(t6c)

   seed             comparison  baseline_correct  arm_correct     n  \
0    42  baseline vs toplosses              1512         1504  1550   
1    42     baseline vs random              1512         1495  1550   
2    43  baseline vs toplosses              1512         1505  1550   
3    43     baseline vs random              1512         1507  1550   
4    44  baseline vs toplosses              1501         1505  1550   
5    44     baseline vs random              1501         1508  1550   
6    45  baseline vs toplosses              1507         1499  1550   
7    45     baseline vs random              1507         1495  1550   
8    46  baseline vs toplosses              1502         1495  1550   
9    46     baseline vs random              1502         1493  1550   

   b_baseline_only  c_arm_only  mcnemar_exact_p  acc_diff  boot_ci_low  \
0               19          11         0.200488 -0.005161    -0.011613   
1               23           6         0.002316 -0.010968    -0.018065

## 9. Per-class table (Table 8) and confusion matrices (Figure 10)

In [12]:
def per_class_of(r):
    return metrics_from_cm(r["cm"])[1].set_index("class")

pc_long = []
for seed in SEEDS_MAIN:
    for arm in ARMS:
        pc_long.append(per_class_of(REF[(arm, seed)]).assign(arm=arm, seed=seed).reset_index())
pc_long = pd.concat(pc_long)
pc_long.to_csv(os.path.join(RESULTS, "table_per_class_all_runs.csv"), index=False)

MET = ["precision", "recall", "specificity", "f1"]
mean_pc = pc_long.groupby(["arm", "class"])[MET].mean()
b_pc, r_pc = mean_pc.loc["baseline"], mean_pc.loc["toplosses"]
t8 = pd.DataFrame(index=CLASSES)
for m in MET:
    t8[f"{m}_baseline"] = b_pc[m]
    t8[f"{m}_refined"] = r_pc[m]
    t8[f"{m}_delta"] = r_pc[m] - b_pc[m]
t8.index.name = "class"
t8.to_csv(os.path.join(RESULTS, "table_per_class_baseline_vs_refined.csv"))
deltas = t8[[f"{m}_delta" for m in MET]].values.ravel()
EPS = 1e-9
sign = {"positive": int((deltas > EPS).sum()), "negative": int((deltas < -EPS).sum()),
        "zero": int((np.abs(deltas) <= EPS).sum()), "total": int(deltas.size)}
# sign pattern per seed as well (single-run deltas)
sign_seed = []
for seed in SEEDS_MAIN:
    a = per_class_of(REF[("baseline", seed)])[MET].values.ravel()
    b = per_class_of(REF[("toplosses", seed)])[MET].values.ravel()
    d = b - a
    sign_seed.append({"seed": seed, "positive": int((d > EPS).sum()), "negative": int((d < -EPS).sum()),
                      "zero": int((np.abs(d) <= EPS).sum())})
pd.DataFrame([{"scope": f"mean over {len(SEEDS_MAIN)} seeds", **sign}] +
             [{"scope": f"seed {s['seed']}", **s, "total": deltas.size} for s in sign_seed]).drop(
    columns=["seed"], errors="ignore").to_csv(os.path.join(RESULTS, "table_per_class_sign_pattern.csv"), index=False)
print(t8.round(4)); print("sign pattern (mean over seeds):", sign)


def plot_cm(ax, cm, title):
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    lab = [c.replace("_", " ") for c in CLASSES]
    ax.set_xticklabels(lab, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(lab, fontsize=7)
    thr = cm.max() / 2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if cm[i, j]:
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=6.5,
                        color="white" if cm[i, j] > thr else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title, fontsize=10)

s0m = SEEDS_MAIN[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, arm, ttl in zip(axes, ["baseline", "toplosses"], ["Baseline", "Top Losses refined"]):
    r = REF[(arm, s0m)]
    plot_cm(ax, r["cm"], f"ResNet50 {ttl} (seed {s0m}): {r['correct']}/{r['n']}")
plt.tight_layout()
save_fig(fig, "fig10_confusion_matrices_baseline_vs_refined.png")

for (n, s), r in CMP.items():
    fig, ax = plt.subplots(figsize=(7, 6.5))
    plot_cm(ax, r["cm"], f"{n} (seed {s}): {r['correct']}/{r['n']}")
    save_fig(fig, f"cm_{n}_s{s}.png")

                          precision_baseline  precision_refined  \
class                                                             
bacterial_leaf_blight                 0.9547             0.9363   
bacterial_leaf_streak                 0.9826             0.9761   
bacterial_panicle_blight              0.9882             0.9770   
blast                                 0.9705             0.9739   
brown_spot                            0.9722             0.9801   
dead_heart                            0.9953             0.9962   
downy_mildew                          0.9230             0.9138   
hispa                                 0.9779             0.9780   
normal                                0.9648             0.9503   
tungro                                0.9773             0.9759   

                          precision_delta  recall_baseline  recall_refined  \
class                                                                        
bacterial_leaf_blight             -0.01

## 10. Figures 3-9: model comparison and refinement arms

In [13]:
METRIC_FIGS = [("accuracy", "Accuracy", "fig3_accuracy.png"), ("macro_precision", "Macro precision", "fig4_precision.png"),
               ("macro_recall", "Macro recall", "fig5_recall.png"), ("macro_specificity", "Macro specificity", "fig6_specificity.png"),
               ("macro_f1", "Macro F1-score", "fig7_f1.png")]
order = t3s.index.tolist()
for col, lab, fn in METRIC_FIGS:
    fig, ax = plt.subplots(figsize=(8, 4))
    v = t3s.loc[order, col].values * 100
    ax.bar(range(len(order)), v, color=["#C44E52" if m == "ResNet50" else "#4C72B0" for m in order])
    for i, x in enumerate(v):
        txt = f"{x:.2f}" + (f"\n({t3s.loc[order[i], 'correct_over_n']})" if col == "accuracy" else "")
        ax.text(i, x + 0.3, txt, ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=20)
    ax.set_ylabel(f"{lab} (%)"); ax.set_ylim(max(0, v.min() - 10), 100 + 4)
    ax.set_title(f"{lab} on the held-out test set (n = {N_TEST})")
    save_fig(fig, fn)

fig, ax = plt.subplots(figsize=(11, 4.5))
wdt = 0.16
for k, (col, lab, _) in enumerate(METRIC_FIGS):
    ax.bar(np.arange(len(order)) + (k - 2) * wdt, t3s.loc[order, col].values * 100, wdt, label=lab)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_ylabel("Score (%)"); ax.set_ylim(max(0, t3s[[c for c, _, _ in METRIC_FIGS]].values.min() * 100 - 10), 101)
ax.legend(ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0))
save_fig(fig, "fig8_all_metrics.png")

fig, ax = plt.subplots(figsize=(7, 4.2))
cols = {"baseline": "#4C72B0", "toplosses": "#C44E52", "random": "#8C8C8C"}
names = {"baseline": "Baseline", "toplosses": f"Top Losses removal (k={N_REMOVE})", "random": f"Random removal (k={N_REMOVE})"}
for s_ in SEEDS_MAIN:   # one grey line per seed, connecting the three arms
    v = [REF[(a_, s_)]["accuracy"] * 100 for a_ in ARMS]
    ax.plot(range(3), v, color="grey", lw=0.8, alpha=0.6, zorder=1)
    ax.scatter(range(3), v, c=[cols[a_] for a_ in ARMS], s=22, zorder=2)
for k, arm in enumerate(ARMS):
    v = ref_df[ref_df.arm == arm].accuracy.values * 100
    sd = v.std(ddof=1) if len(v) > 1 else 0.0
    ax.errorbar(k + 0.18, v.mean(), yerr=sd, fmt="D", color="black", ms=6, capsize=4, zorder=3)
    ax.text(k + 0.26, v.mean(), f"{v.mean():.2f} ± {sd:.2f}", va="center", fontsize=8)
ax.set_xticks(range(3)); ax.set_xticklabels([names[a] for a in ARMS], fontsize=8)
ax.set_xlim(-0.4, 2.9)
ax.set_ylabel("Test accuracy (%)")
ax.set_title(f"ResNet50 refinement arms: {len(SEEDS_MAIN)} seeds (grey lines), mean ± SD (black)", fontsize=10)
save_fig(fig, "fig9_refinement_gains.png")

saved /kaggle/working/results/figures/fig3_accuracy.png
saved /kaggle/working/results/figures/fig4_precision.png
saved /kaggle/working/results/figures/fig5_recall.png
saved /kaggle/working/results/figures/fig6_specificity.png
saved /kaggle/working/results/figures/fig7_f1.png
saved /kaggle/working/results/figures/fig8_all_metrics.png
saved /kaggle/working/results/figures/fig9_refinement_gains.png


## 11. Figure 11: the removed Top Losses images (first seed)

In [14]:
rm0 = removed_df[removed_df.seed == SEEDS_MAIN[0]]
uid2path = dict(zip(data.uid, data.path))
fig, axes = plt.subplots(1, len(rm0), figsize=(3.2 * len(rm0), 4))
axes = np.atleast_1d(axes)
for ax, (_, r) in zip(axes, rm0.iterrows()):
    ax.imshow(Image.open(uid2path[r.uid]).convert("RGB"))
    ax.set_title(f"#{r['rank']} {r.uid.split('/')[-1]}\nlabel: {r.true_label}\npred: {r.pred_label}\n"
                 f"loss {r.loss:.2f}, conf {r.pred_conf:.2f}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
save_fig(fig, "fig11_removed_top_losses_images.png")

saved /kaggle/working/results/figures/fig11_removed_top_losses_images.png


## 12. Figure 12: Grad-CAM, baseline vs refined (ResNet50 layer4, one test image per class)

Both heatmaps are computed for the true class of the image; panel titles give each model's prediction and confidence.

In [15]:
class GradCAM:
    """Forward hook on the target layer stores the activation and registers a tensor hook on that
    output to capture its gradient (no module full-backward hook)."""
    def __init__(self, model, layer):
        self.model, self.act, self.grad = model, None, None
        self.h = layer.register_forward_hook(self._fwd)

    def _fwd(self, module, inp, out):
        self.act = out
        out.register_hook(self._save_grad)

    def _save_grad(self, g):
        self.grad = g

    def __call__(self, x, cls=None):
        """Heatmap for target class `cls` (default: predicted class). Returns cam, predicted class, its probability."""
        self.model.eval()
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            logits = self.model(x)
            probs = F.softmax(logits, 1)
            pred = int(logits.argmax(1))
            if cls is None:
                cls = pred
            self.model.zero_grad(set_to_none=True)
            logits[0, cls].backward()
        w = self.grad.mean(dim=(2, 3), keepdim=True)              # alpha_k^c
        cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), pred, float(probs[0, pred])

    def remove(self):
        self.h.remove()


m_base = load_model(f"ResNet50_baseline_s{s0m}")
m_ref = load_model(f"ResNet50_toplosses_s{s0m}")
cam_b, cam_r = GradCAM(m_base, m_base.layer4), GradCAM(m_ref, m_ref.layer4)

te = data.iloc[IDX_TE]
pick = [te[te.y == k].sort_values("uid").index[0] for k in range(NUM_CLASSES) if (te.y == k).any()]
gc_rows = []
fig, axes = plt.subplots(len(pick), 3, figsize=(7.5, 2.5 * len(pick)))
axes = np.atleast_2d(axes)
for row, i in enumerate(pick):
    img = CACHE[i]
    x = ((torch.from_numpy(img).permute(2, 0, 1).float() / 255 - MEAN) / STD).unsqueeze(0).to(DEVICE)
    hb, pb, cb = cam_b(x, cls=int(Y[i]))   # both maps explain the TRUE class, so they are comparable
    hr, pr, cr = cam_r(x, cls=int(Y[i]))
    gc_rows.append({"uid": UIDS[i], "true": CLASSES[Y[i]], "baseline_pred": CLASSES[pb], "baseline_conf": cb,
                    "refined_pred": CLASSES[pr], "refined_conf": cr})
    np.save(os.path.join(RUNS_DIR, f"gradcam_{CLASSES[Y[i]]}.npy"), np.stack([hb, hr]))
    axes[row, 0].imshow(img); axes[row, 0].set_title(f"{CLASSES[Y[i]]}\n{UIDS[i].split('/')[-1]}", fontsize=7)
    axes[row, 1].imshow(img); axes[row, 1].imshow(hb, cmap="jet", alpha=0.45)
    axes[row, 1].set_title(f"Baseline: {CLASSES[pb]} ({cb:.2f})", fontsize=7)
    axes[row, 2].imshow(img); axes[row, 2].imshow(hr, cmap="jet", alpha=0.45)
    axes[row, 2].set_title(f"Refined: {CLASSES[pr]} ({cr:.2f})", fontsize=7)
    for a in axes[row]:
        a.axis("off")
plt.tight_layout()
save_fig(fig, "fig12_gradcam_baseline_vs_refined.png")
cam_b.remove(); cam_r.remove()
pd.DataFrame(gc_rows).to_csv(os.path.join(RESULTS, "gradcam_examples.csv"), index=False)
del m_base, m_ref

saved /kaggle/working/results/figures/fig12_gradcam_baseline_vs_refined.png


## 13. Run summary (values for the manuscript placeholders) and results_pack.zip

In [16]:
b0 = REF[("baseline", s0m)]; r0 = REF[("toplosses", s0m)]
t6_tl = t6[t6.comparison == "baseline vs toplosses"]
rank_resnet = int(t3s.loc["ResNet50", "rank"])
summary = {
    "N_SRC": int(N_SRC), "N_DUP": int(N_DUP), "N_CONFLICT_FILES": N_CONFLICT_FILES, "N_UNIQUE": int(N_UNIQUE),
    "N_TRAIN": int(N_TRAIN), "N_VAL": int(N_VAL), "N_TEST": int(N_TEST), "N_NEAR_CROSS": int(N_NEAR_CROSS),
    "N_SEEDS": len(SEEDS_MAIN), "N_REMOVE": N_REMOVE,
    "first_seed": s0m,
    "K_BASE": int(b0["correct"]), "ACC_BASE": b0["accuracy"],
    "K_XAI": int(r0["correct"]), "ACC_XAI": r0["accuracy"],
    "ACC_BASE_MEAN": float(agg.loc["baseline", "acc_mean"]), "ACC_BASE_SD": float(agg.loc["baseline", "acc_sd"]),
    "ACC_XAI_MEAN": float(agg.loc["toplosses", "acc_mean"]), "ACC_XAI_SD": float(agg.loc["toplosses", "acc_sd"]),
    "MEAN_RAND": float(agg.loc["random", "acc_mean"]), "SD_RAND": float(agg.loc["random", "acc_sd"]),
    "P_MCN": float(t6_tl[t6_tl.seed == s0m].mcnemar_exact_p.iloc[0]),
    "P_MCN_RANGE": [float(t6_tl.mcnemar_exact_p.min()), float(t6_tl.mcnemar_exact_p.max())],
    "SEEDS_TOPLOSSES_SIGNIFICANT": int(t6_tl["significant_0.05"].sum()),
    "WILCOXON_P": w_p, "WILCOXON_MIN_REACHABLE_P": min_p,
    "SEEDS_TOPLOSSES_BEAT_RANDOM": int((diff > 0).sum()),
    "NOT_SIGNIFICANT": bool(t6_tl["significant_0.05"].sum() == 0),
    "RESNET50_RANK": rank_resnet,
    "RANK_STATEMENT": f"ResNet50 ranked {rank_resnet} of {len(MODELS)} models by test accuracy "
                      f"(best: {t3s.index[0]}, {t3s.correct_over_n.iloc[0]}).",
    "PER_CLASS_SIGN_MEAN": sign,
    "split_fracs": SPLIT_FRACS, "split_seed": SPLIT_SEED,
    "config": {"epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE, "img_size": IMG_SIZE, "lr": LR,
               "optimizer": "Adam", "scheduler": "CosineAnnealingLR(T_max=EPOCHS)", "amp": USE_AMP,
               "seeds_main": SEEDS_MAIN, "seeds_compare": SEEDS_COMPARE, "n_boot": N_BOOT},
    "env": ENV, "total_minutes": (time.time() - T_START) / 60,
}
json.dump(summary, open(os.path.join(RESULTS, "run_summary.json"), "w"), indent=2, default=float)
print(json.dumps(summary, indent=2, default=float))

zpath = os.path.join(WORK_DIR, "results_pack_QUICKTEST.zip" if QUICK_TEST else "results_pack.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RESULTS):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, WORK_DIR))
print(f"wrote {zpath} ({os.path.getsize(zpath) / 1e6:.1f} MB). Download it from the Output tab.")

{
  "N_SRC": 10407,
  "N_DUP": 74,
  "N_CONFLICT_FILES": 0,
  "N_UNIQUE": 10333,
  "N_TRAIN": 7749,
  "N_VAL": 1034,
  "N_TEST": 1550,
  "N_NEAR_CROSS": 1019,
  "N_SEEDS": 5,
  "N_REMOVE": 5,
  "first_seed": 42,
  "K_BASE": 1512,
  "ACC_BASE": 0.9754838709677419,
  "K_XAI": 1504,
  "ACC_XAI": 0.9703225806451613,
  "ACC_BASE_MEAN": 0.9721290322580645,
  "ACC_BASE_SD": 0.003395534804371815,
  "ACC_XAI_MEAN": 0.9687741935483871,
  "ACC_XAI_SD": 0.0028707865111338455,
  "MEAN_RAND": 0.967483870967742,
  "SD_RAND": 0.004687974741828376,
  "P_MCN": 0.20048842206597334,
  "P_MCN_RANGE": [
    0.20048842206597334,
    0.5846647117286922
  ],
  "SEEDS_TOPLOSSES_SIGNIFICANT": 0,
  "WILCOXON_P": 0.625,
  "WILCOXON_MIN_REACHABLE_P": 0.0625,
  "SEEDS_TOPLOSSES_BEAT_RANDOM": 3,
  "NOT_SIGNIFICANT": true,
  "RESNET50_RANK": 1,
  "RANK_STATEMENT": "ResNet50 ranked 1 of 7 models by test accuracy (best: ResNet50, 1512/1550).",
  "PER_CLASS_SIGN_MEAN": {
    "positive": 12,
    "negative": 25,
    "zero"